In [ ]:
from core import altopt, plot_altopt
from core import DataContext, ParameterContext
from core import get_final_interpolator, load_experiment, get_predictions_batch
import os
from scipy.optimize import curve_fit
import numpy as np
import pandas as pd
import cupy as cp
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'serif', 'font.size': 12})

In [ ]:
DATA_DIR = r"DataFiles_to_Dinesh_Pranav\Data_files\Simulation\Dataset_10"
ilist = [0,1,2,3]
for k in [ "Gaussian"]:
    for sigma_noise in [0.5]:
        for prior in [ 2.0]:
            for est_first in [False, True]:
                for i in ilist:
                    for j in range(0,10,2):
                        # if (not est_first and prior == 1.5 and sigma_noise == 0.4) or (est_first and prior == 1.5 and sigma_noise == 0.4 and i in [0,1]):
                        #     continue
                        curr_time = 15.0 + j
                        DATASET3 = DataContext(
                            sim_time = os.path.join(DATA_DIR, "t_array.csv"),
                            sim_freq = os.path.join(DATA_DIR, "delz_MHz.csv"),
                        sim_by = os.path.join(DATA_DIR, "dely_MHz.csv"),
                        sim_intensities = sorted([
                            os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.startswith("transH_t_vs_Bz_alpha2500_delx_0_dely_") and f.endswith(".csv")
                        ], key=lambda x: float(x.split("_dely_")[1].split(".csv")[0])),
                        exp_time = r"DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\t_array_ Copy.csv",
                        exp_freq_axis = r"DataFiles_to_Dinesh_Pranav\Data_files\Experiment\26_03_2026_CW_data\Bz_Y_MHz.csv",
                        exp_data = f"DataFiles_to_Dinesh_Pranav\\Data_files\\Experiment\\26_03_2026_CW_data\\Bz_V_Readout_By_{i}.csv",
                        save_path = f"D:\\Pranav_and_Dinesh\\Dinesh\\Results\\Dataset_10\\03_June\\Prior_Test_{prior}\\Linear_Interpolation\\Est_First_Z_{est_first}\\{k}_Likelihood\\Sigma_{sigma_noise}\\Test_{(j//2)}_Dataset_10_By_{i}_from_{curr_time}_microseconds_Randomised_Start_Time_sigma_noise_{sigma_noise}",
                        interpolator=r"DataFiles_to_Dinesh_Pranav\Data_files\Simulation\Dataset_10\gpu_sim_interpolator_new_normalisation_linear_spline_0.3_threshold.npz",
                        Sim_Aligned = True,
                        Exp_Aligned = False,
                        sim_pulse_thresh=0.3, 
                        exp_pulse_thresh=0.3
                        )
                        DEFAULT_ALTOPT_PARAMS = ParameterContext(
                            num_iter=10,
                            curr_time = curr_time,
                            max_time = 70.0,
                            tol_bz=1e-20,
                            tol_by=1e-20,
                            t_step = 0.2,   
                            fixed_by_estimate=0.0,
                            fixed_bz_estimate=0.0,
                            B_unk_bound_transverse_upper=0.5,
                            B_unk_bound_longitudinal = prior,
                            print_plot = False,
                            Est_First_Z= est_first,
                            likelihood_mode_longitudinal=k,
                            likelihood_mode_transverse=k,
                            sigma_noise_longitudinal = sigma_noise,
                            sigma_noise_transverse = sigma_noise
                            )
                        sign = "plus" if DEFAULT_ALTOPT_PARAMS.fixed_bz_estimate >= 0 else "minus"
                        Test = r"Dataset_10_Starting with" + str(DEFAULT_ALTOPT_PARAMS.curr_time) + r" us_T Step " + str(DEFAULT_ALTOPT_PARAMS.t_step) + r" us_" + sign + r"_" + str(np.abs(DEFAULT_ALTOPT_PARAMS.fixed_bz_estimate)) + r"_initial estimate"
                        save_path = sign + r"_" + str(np.abs(DEFAULT_ALTOPT_PARAMS.fixed_bz_estimate)) + r"_initial_estimate"
                        DEFAULT_ALTOPT_PARAMS.Test = Test

                        results = altopt(DATASET3, DEFAULT_ALTOPT_PARAMS)
                        plot_altopt(DATASET3, *results)